In [1]:
from lib2to3.pgen2.tokenize import Bracket

import torch
from torch import nn

torch.manual_seed(23)

In [8]:
## 2. 从稀疏表示到可学习的稠密表示
vocab = {"猫": 0, "狗": 1, "追": 2, "球": 3}
# 这仍是一个可训练参数表：初始化时向量通常是随机的，只有参与训练后才可能学到任务有用的结构。若只是拿随机 embedding 做相似度，结果没有可靠语义。
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=3)
ids = torch.tensor([vocab["猫"], vocab["狗"]])
print(f"ids: {ids}")
vectors = embedding(ids)
print(vectors.shape)  # [2, 3]
print(embedding.weight.shape)  # [词表大小, 向量维度]

ids: tensor([0, 1])
torch.Size([2, 3])
torch.Size([4, 3])


In [9]:
import torch
from torch import nn

corpus = "我 喜欢 自然 语言 处理 我 喜欢 深度 学习".split()
vocab = {word: i for i, word in enumerate(sorted(set(corpus)))}
encoded = [vocab[word] for word in corpus]

def make_cbow_examples(ids, radius=2):
    examples = []
    # 只保留完整窗口，保证每个样本上下文长度相同，便于直接组成 Tensor。
    for pos in range(radius, len(ids) - radius):
        context = [ids[j] for j in range(pos-radius, pos+radius+1) if j != pos]
        examples.append((context, ids[pos]))
    return examples

class CBOW(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.input_embedding = nn.Embedding(vocab_size, dim)
        self.output = nn.Linear(dim, vocab_size)

    def forward(self, contexts):  # contexts: [B, C]
        hidden = self.input_embedding(contexts).mean(dim=1)  # [B, D]
        return self.output(hidden)  # [B, V] logits

examples = make_cbow_examples(encoded)
contexts = torch.tensor([ctx for ctx, _ in examples])
targets = torch.tensor([target for _, target in examples])
model = CBOW(len(vocab), dim=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for _ in range(100):
    logits = model(contexts)
    loss = nn.functional.cross_entropy(logits, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("loss:", loss.item())

loss: 0.00307597778737545


In [10]:
import torch
from torch import nn
import torch.nn.functional as F

class SkipGramNegativeSampling(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)
        self.context = nn.Embedding(vocab_size, dim)

    def forward(self, center_ids, positive_ids, negative_ids):
        # center_ids, positive_ids: [B]; negative_ids: [B, K]
        v = self.center(center_ids)                 # [B, D]
        u_pos = self.context(positive_ids)           # [B, D]
        u_neg = self.context(negative_ids)           # [B, K, D]
        pos_score = (v * u_pos).sum(-1)              # [B]
        neg_score = torch.bmm(u_neg, v.unsqueeze(-1)).squeeze(-1)  # [B, K]
        loss = -F.logsigmoid(pos_score).mean() - F.logsigmoid(-neg_score).sum(1).mean()
        return loss

model = SkipGramNegativeSampling(vocab_size=100, dim=32)
center_ids = torch.tensor([4, 9, 3])
positive_ids = torch.tensor([5, 8, 2])
negative_ids = torch.randint(0, 100, (3, 5))  # 示范用均匀采样；真实训练应按词频采样
loss = model(center_ids, positive_ids, negative_ids)
loss.backward()
print(loss.item())

11.730388641357422


In [11]:
from transformers import AutoTokenizer, AutoModel

### # 在现代NLP任务中，通常不会从0开始训练Embedding模型，而是加载已经训练好的预训练Embedding

In [16]:
# 在现代NLP任务中，通常不会从0开始训练Embedding模型，而是加载已经训练好的预训练Embedding

# 使用预训练的 BERT 中文模型，将一段中文文本转换成 Token ID，再通过 BERT 模型计算每个 Token 对应的上下文语义向量（Contextual Embedding）。

import torch
from transformers import AutoTokenizer, AutoModel

# 指定要加载的预训练模型
model_name = "bert-base-chinese"  # 中文示例，需联网下载权重
# model_name = "Qwen/Qwen2.5-0.5B"
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 加载嵌入模型
model = AutoModel.from_pretrained(model_name)
# 准备要测试的分词数据
text = "苹果发布了新产品。"
# 进行分词，得到测试数据的嵌入id， “pt”表示Pytorch Tensor，truncation=True，超出最大长度时，自动截断超出的长度（BERT Base最长512个token）
seq_token = tokenizer(text, return_tensors="pt", truncation=True)
# seq_token 里面包含三个字典
# input_ids	        每个 Token 对应的整数 ID
# token_type_ids	标识 Token 属于哪一个句子，常用于区分句子 A 和 B
# attention_mask	标识哪些位置是真实 Token，哪些位置是 Padding
print(seq_token)
# 开启评估模式，并且禁用梯度计算，因此从中可以得知预训练模型是还可以自己再次训练的
model.eval()
with torch.no_grad():
    # 将 Token 输入 BERT 模型，将 Tokenizer 处理后的输入传递给 BERT 模型，执行前向传播。最终得到预训练模型嵌入的输出向量
    output = model(**seq_token)
token_vectors = output.last_hidden_state
print(seq_token["input_ids"].shape)
print(token_vectors.shape)  # [B, L, D]
# Tokenizer 会把文本映射成模型训练时约定的 token ID，并加入特殊 token。必须使用与 checkpoint 匹配的 tokenizer；自行换分词器会导致 ID 语义错位，即使形状看起来正常也没有意义。

# ：查看每个 Token 对应的 ID
# [CLS]：Classification Token：分类标记，通常放在输入文本的最前面。
# [SEP]：Separator Token：分隔符，主要用于标记句子结束或分隔不同句子。
tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
print(tokens)
print(batch["input_ids"][0])

# 获取嵌入模型的配置
embedding = model.get_input_embeddings()
print(embedding)
print("词表大小:", embedding.num_embeddings)
print("Embedding 维度:", embedding.embedding_dim)

Could not reach the Hub (Server disconnected without sending a response.). Using cached commit hash for 'bert-base-chinese'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'input_ids': tensor([[ 101, 5741, 3362, 1355, 2357,  749, 3173,  772, 1501,  511,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 11])
torch.Size([1, 11, 768])
['[CLS]', '苹', '果', '发', '布', '了', '新', '产', '品', '。', '[SEP]']
tensor([ 101, 5741, 3362, 1355, 2357,  749, 3173,  772, 1501,  511,  102])
Embedding(21128, 768, padding_idx=0)
词表大小: 21128
Embedding 维度: 768


In [44]:
# 加载预训练的嵌入模型

import torch
from transformers import AutoTokenizer, AutoModel

# 定义预训练模型名称
model_name = "bert-base-chinese"
# 加载预训练分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 加载预训练嵌入模型
model = AutoModel.from_pretrained(model_name)
# 准备预测数据
text = "加载预训练的嵌入模型"
# 获取预测数据的token_ids
text_token = tokenizer(text, return_tensors="pt", truncation=True)
print(text_token["input_ids"].shape) # [B, L]
# 开启模型评估模式进行嵌入向量推理
model.eval()
# 通过text_token里面的input_ids的嵌入token id来得到嵌入向量
with torch.no_grad():
    # 解包 text_token
    output = model(**text_token)
# 获取整个测试文本的隐藏层输出 嵌入表示
token_embeddings = output.last_hidden_state
print(token_embeddings.shape) # [B, L, D], D 表示嵌入向量维度


# 拿到嵌入向量之后，将 token_embeddings 直接输入 语言模型（RNN，BERT...）里面即可
print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 12])
torch.Size([1, 12, 768])



In [25]:
# 将 Token 和嵌入向量对应起来
# text_token["input_ids"][0].shape
idx_to_tokens = tokenizer.convert_ids_to_tokens(text_token["input_ids"][0])
print(idx_to_tokens)

# 可以获取每一个token的嵌入
# for embed in token_embeddings[0]:
#     print(embed.shape)


['[CLS]', '加', '载', '预', '训', '练', '的', '嵌', '入', '模', '型', '[SEP]']
